# Tutorial 3: Making a Long Training Run Survive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/03-train-tiny-model.ipynb)

**Reading. About 15 minutes; it runs in under a minute.**

A real training run takes hours, and a Colab session can disconnect in the middle of one. This
notebook makes a run that survives that: it saves as it goes, picks up where it stopped when you
run it again, and leaves behind a model you can load the next day. It trains on 2,000 real
sentence pairs for a few epochs, far too little to translate, because the point is the machinery
around training, which is the same at any size.

By the end you will be able to:

1. train with `train_model` so that a dead session costs nothing: a `TrainingCheckpointer` that resumes, and a `save_dir` that keeps the best model;
2. recover from an interrupted run by running the same cell again;
3. rebuild the saved model in a fresh session, and check that it is the model you trained.

**Running in Google Colab?** Run the next two cells. The first installs TorchLingo; the second
checks the setup and downloads the corpus, about 21 MB. There is nothing to uncomment, and no GPU
is needed.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 7 (*Research Paper Reviews; Intro to Neural Networks*). A head start on **A8**.

In [1]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [2]:
# Check the setup: the TorchLingo and PyTorch versions and the device, printed here so that
# anything wrong shows up now rather than ten cells from now. The corpus is in the repository,
# not the pip package, so setup downloads it if it is missing.
from torchlingo.colab import setup

env = setup(data=["data/example.tsv"])

TorchLingo 0.2.6 (editable checkout) | PyTorch 2.13.0 | device cpu | not in Colab
  data/example.tsv    21.0 MB


In [3]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from torchlingo.checkpoint import load_checkpoint
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

device = env.device

## Part 1: A training and a validation split

2,000 sentence pairs from TED talk transcripts to train on, and the next 200 to validate on. The
validation set is what lets the trainer tell which epoch's model is best, so it is the one worth
keeping.

In [4]:
corpus = pd.read_csv("data/example.tsv", sep="\t", dtype=str, keep_default_na=False)
corpus = corpus.loc[corpus["kind"] == "transcript", ["src", "tgt"]]

train_path = Path("data/tutorial-3-train.tsv")
val_path = Path("data/tutorial-3-val.tsv")
corpus.iloc[:2000].to_csv(train_path, sep="\t", index=False)
corpus.iloc[2000:2200].to_csv(val_path, sep="\t", index=False)

# The validation set must use the training set's vocabularies, never its own.
train_set = NMTDataset(train_path)
val_set = NMTDataset(val_path, src_vocab=train_set.src_vocab, tgt_vocab=train_set.tgt_vocab)
print(f"{len(train_set):,} training pairs, {len(val_set)} validation pairs")
print(f"vocabularies: {len(train_set.src_vocab):,} English, {len(train_set.tgt_vocab):,} Spanish")

2,000 training pairs, 200 validation pairs
vocabularies: 2,507 English, 2,711 Spanish


## Part 2: The model, and everything needed to build it again

Rebuilding a model later takes the same `Config` and the same vocabularies, so keep both in
mind: the saved file will hold only the weights. The sizes here are tiny, so an epoch takes
seconds.

In [5]:
config = Config(
    d_model=64, n_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=128,
    dropout=0.1, batch_size=32, learning_rate=1e-3,
)


def new_model():
    """An untrained model of this notebook's shape, built from the config and vocabularies."""
    return SimpleTransformer(
        src_vocab_size=len(train_set.src_vocab),
        tgt_vocab_size=len(train_set.tgt_vocab),
        config=config,
    ).to(device)


# num_workers=0: worker processes cost more to start than they save on a corpus this size.
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn,
                          num_workers=0)
val_loader = DataLoader(val_set, batch_size=32, collate_fn=collate_fn, num_workers=0)

torch.manual_seed(0)
model = new_model()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

677,847 parameters


## Part 3: Train, saving as you go

Two arguments to `train_model` make a run survive. A8 passes both:

- **`checkpointer`**, a `TrainingCheckpointer`, saves the whole run's progress (model, optimizer,
  epoch) as it goes. Run the cell again after a disconnect and it **resumes**. In A8 it saves to
  Google Drive, because a Colab runtime forgets its own disk when it disconnects; here, a local
  folder.
- **`save_dir`** keeps the model from the epoch with the lowest validation loss, as
  `model_best.pt`. That file is the run's result.

This cell asks for two epochs and stops, as if the session had died there.

In [6]:
RUN_DIR = Path("checkpoints/tutorial-3")


def train(num_epochs):
    """Train toward num_epochs in total, resuming from RUN_DIR if a run is saved there."""
    checkpointer = TrainingCheckpointer("tutorial-3", checkpoint_dir=RUN_DIR / "resume")
    return train_model(
        model, train_loader, val_loader=val_loader, num_epochs=num_epochs,
        device=device, config=config, gradient_clip=1.0, log_every=0,
        save_dir=RUN_DIR / "best", checkpointer=checkpointer,
    )


result = train(num_epochs=2)

[checkpoint] checkpoints -> checkpoints/tutorial-3/resume, 305,409 MB free


Epoch 1/2 | Train: 7.9382 | Val: 7.8978
[checkpoint] saved epoch 0, step 63 (best)
  -> Saved best checkpoint


Epoch 2/2 | Train: 7.8477 | Val: 7.6669
[checkpoint] saved epoch 1, step 126 (best)
  -> Saved best checkpoint


## Part 4: The session died. Run it again.

A disconnect takes the model in memory with it. The next cell stands in for a fresh session: a
new, untrained model object, and the same `train` call asking for four epochs in total. Its
second line reads `resumed from epoch 1`: the checkpointer counts from 0, so that is the end of
the second epoch, and training goes on with epoch 3 of 4. Without the checkpointer, all four
would start again from nothing; on A8 that is hours.

In [7]:
torch.manual_seed(0)
model = new_model()  # what a fresh session has: nothing trained
result = train(num_epochs=4)

print(f"\nvalidation loss by epoch: {[round(v, 3) for v in result.val_losses]}")

[checkpoint] checkpoints -> checkpoints/tutorial-3/resume, 305,385 MB free
[checkpoint] resumed from epoch 1, step 126


Epoch 3/4 | Train: 7.6746 | Val: 7.3554
[checkpoint] saved epoch 2, step 189 (best)
  -> Saved best checkpoint


Epoch 4/4 | Train: 7.4723 | Val: 7.0785
[checkpoint] saved epoch 3, step 252 (best)
  -> Saved best checkpoint

validation loss by epoch: [7.898, 7.667, 7.355, 7.078]


In [8]:
# What the run left on disk: the resumable state, and the best model.
for path in sorted(RUN_DIR.rglob("*.pt")):
    print(f"{path}  ({path.stat().st_size / 1e6:.1f} MB)")

checkpoints/tutorial-3/best/model_best.pt  (8.2 MB)
checkpoints/tutorial-3/resume/best.pt  (8.2 MB)
checkpoints/tutorial-3/resume/latest.pt  (8.2 MB)


## Part 5: Rebuild the model the next day

`model_best.pt` holds the weights, not the model and not the vocabularies. So a fresh session
rebuilds the same pieces and pours the weights in:

1. the vocabularies, from the same training file (building them is deterministic, so they come
   out identical);
2. an empty model, from the same `Config`;
3. the weights, with `load_checkpoint`.

A8's scoring cell does exactly this, so it can score your model a day after training it. The
last line checks the claim rather than trusting it: the rebuilt model and the trained one must
produce the same numbers on the same input.

In [9]:
# 1. The vocabularies, rebuilt from the same training file.
fresh = NMTDataset(train_path)

# 2. An empty model from the same Config and vocabulary sizes, and 3. the saved weights.
rebuilt = SimpleTransformer(
    src_vocab_size=len(fresh.src_vocab), tgt_vocab_size=len(fresh.tgt_vocab), config=config
).to(device)
best = load_checkpoint(result.best_checkpoint, map_location=device)
rebuilt.load_state_dict(best["model_state_dict"])

# The check: same input, same output. Validation loss fell every epoch, so the model still in
# memory is the one that was saved as best.
src, tgt = next(iter(val_loader))
src, tgt = src.to(device), tgt.to(device)
model.eval()
rebuilt.eval()
with torch.no_grad():
    same = torch.allclose(model(src, tgt[:, :-1]), rebuilt(src, tgt[:, :-1]))
print(f"vocabularies identical: {fresh.tgt_vocab.idx2token == train_set.tgt_vocab.idx2token}")
print(f"rebuilt model matches the trained one: {same}")

vocabularies identical: True
rebuilt model matches the trained one: True


## What's next

This model cannot translate yet: four epochs on 2,000 pairs is a few minutes' worth of a run
that needs hours. [Tutorial 7](07-real-translations.ipynb) loads one that trained for 36 epochs
on 64,000 pairs, and shows what it can and cannot do.